# 方程求解 / Equation Solving

---

求解代数方程组是科学和技术领域中的常见问题。相对于线性方程组，非线性方程组通常较难求解。线性方程是求解非线性问题局部近似解的重要工具。例如，考虑某个展开点附近非常小的变动，非线性系统通常可以使用展开点附近的线性西戎来近似。对于非线性问题的全局分析，一般需要采用迭代方式来逐步构建对解越来越精确的估计

Solving systems of algebraic equations is a common problem in science and engineering. Compared with linear systems, nonlinear systems are usually more difficult to solve. Linear equations are an important tool for finding local approximate solutions to nonlinear problems. For example, when considering very small perturbations around an expansion point, nonlinear systems can often be approximated by a linear system near that point. For global analysis of nonlinear problems, an iterative method is typically used to progressively build increasingly accurate estimates of the solution.

本部分我们将使用SymPy对方程进行符号化求解，使用SciPy的线性代数模块来对方程组进行数值求解。为了解决非线性问题，我们将使用SciPy的optimize模块的`root-finding`函数。

In this part, we will use SymPy to solve equations symbolically and use SciPy's linear algebra module to solve systems of equations numerically. To tackle nonlinear problems, we will use the `root-finding` functions of SciPy's optimize module.

<!-- bilingual -->

## 导入模块 / Importing Modules

---

<!-- bilingual -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy import linalg as la
from scipy import optimize

import sympy
sympy.init_printing()

In [ ]:
%reload_ext version_information
%version_information numpy, matplotlib, scipy, sympy

## 线性方程组 / Systems of Linear Equations

---

我们已经在SymPy库中使用了线性代数功能。NumPy和SciPy也有线性代数模块，分别是numpy.linalg和scipy.linalg，它们都为数值问题提供线性代数问题。

We have already used linear algebra functionality in the SymPy library. NumPy and SciPy also have linear algebra modules, numpy.linalg and scipy.linalg, which provide linear algebra for numerical problems.

[线性方程组](https://zh.m.wikipedia.org/wiki/%E7%BA%BF%E6%80%A7%E6%96%B9%E7%A8%8B%E7%BB%84)是数学方程组的一种，它符合以下的形式：

A [system of linear equations](https://en.wikipedia.org/wiki/System_of_linear_equations) is a type of system of mathematical equations that has the following form:

$$ \begin{cases}
a_{1,1}x_{1} + a_{1,2}x_{2} + \cdots + a_{1,n}x_{n}=  b_{1} \\
a_{2,1}x_{1} + a_{2,2}x_{2} + \cdots + a_{2,n}x_{n}=  b_{2} \\
\vdots \quad \quad \quad \vdots \\
a_{m,1}x_{1} + a_{m,2}x_{2} + \cdots + a_{m,n}x_{n}=  b_{m} 
\end{cases} $$

这是一个包含m个方程、n个未知数的线性方程组。处理线性方程组时，将其写为矩阵的形式会更方便：

This is a linear system with m equations and n unknowns. When working with linear systems, it is more convenient to write them in matrix form:

$$ \mathbf{A} \mathbf{x} = \mathbf{b} $$

其中

where

$$\mathbf A=
\begin{bmatrix}
a_{1,1} & a_{1,2} & \cdots & a_{1,n} \\
a_{2,1} & a_{2,2} & \cdots & a_{2,n} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m,1} & a_{m,2} & \cdots & a_{m,n}
\end{bmatrix},\quad
\mathbf{x}=
\begin{bmatrix}
x_1 \\
x_2 \\
\vdots \\
x_n
\end{bmatrix},\quad
\mathbf{b}=
\begin{bmatrix}
b_1 \\
b_2 \\
\vdots \\
b_m
\end{bmatrix}$$

根据矩阵$A$的性质，解$x$可能存在也可能不存在。如果方程组中$m < n$，那么称之为欠定（underdetermined）方程组，不能完全确定唯一解。如果 $m < n$，称之为超定（overdetermined）方程组。这通常会带来约束冲突，导致解不存在。

Depending on the properties of the matrix $A$, a solution $x$ may or may not exist. If $m < n$, the system is called underdetermined and cannot completely determine a unique solution. If $m > n$, it is called overdetermined. This usually leads to conflicting constraints and no solution.

<!-- bilingual -->

### 方形方程组 / Square Systems

方形方程组（$m = n$）是一个最重要的特例。当方程的数目等于未知数的数目，可能存在唯一解。

A square system ($m = n$) is the most important special case. When the number of equations equals the number of unknowns, a unique solution may exist.

有唯一解的前提是矩阵$A$必须是非奇异的，也就是$A$存在逆矩阵，解可以写为$x = A^{-1}b$。

The prerequisite for a unique solution is that the matrix $A$ must be non-singular; that is, $A$ has an inverse, and the solution can be written as $x = A^{-1}b$.

如果行列式为0，即$detA = 0$，则方程组无解或有无穷多解。

If the determinant is 0, i.e. $\det A = 0$, then the system has no solution or infinitely many solutions.

对于秩不足的矩，即$rank(A) < n$，矩阵中有的行或者列可以便是成其它行或者列的线性组合，方程组实际上是欠定的。

For a rank-deficient matrix, i.e. $\text{rank}(A) < n$, some rows or columns of the matrix can be written as linear combinations of the other rows or columns, and the system is in fact underdetermined.

当$A$满秩时，一定存在解，但是可能无法精确计算解。矩阵的条件数$cond(A)$给出了线性方程组好坏的条件。条件数接近1，方程组是条件良态；条件数很大，方程组是条件病态的。对于病态方程组，即使向量$b$发生非常微小的扰动，也会让解$x$产生很大的误差。这在使用浮点数的数值解中尤其需要注意，因为浮点数只是实数的近似值。

When $A$ has full rank, a solution is guaranteed to exist, but it may not be computable exactly. The matrix condition number $\text{cond}(A)$ provides a measure of how well-conditioned the linear system is. A condition number close to 1 means the system is well-conditioned; a very large condition number means the system is ill-conditioned. For ill-conditioned systems, even a tiny perturbation of the vector $b$ can cause a large error in the solution $x$. This is particularly important to keep in mind in numerical solutions that use floating-point numbers, since floating-point values are only approximations of real numbers.

<!-- bilingual -->

例如，对于如下包含两个线性方程的方程组：

For example, consider the following system of two linear equations:

$$\begin{cases} 
2 x_1 + 3 x_2 = 4 \\
5 x_1 + 4 x_2 = 3
\end{cases}$$

<!-- bilingual -->

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x1 = np.linspace(-4, 2, 100)

x2_1 = (4 - 2 * x1)/3
x2_2 = (3 - 5 * x1)/4

ax.plot(x1, x2_1, 'r', lw=2, label=r"$2x_1+3x_2-4=0$")
ax.plot(x1, x2_2, 'b', lw=2, label=r"$5x_1+4x_2-3=0$")

A = np.array([[2, 3], [5, 4]])
b = np.array([4, 3])
x = la.solve(A, b)

ax.plot(x[0], x[1], 'ko', lw=2)
ax.annotate("The intersection point of\nthe two lines is the solution\nto the equation system",
            xy=(x[0], x[1]), xycoords='data',
            xytext=(-120, -75), textcoords='offset points', 
            arrowprops=dict(arrowstyle="->", connectionstyle="arc3, rad=-.3"))

ax.set_xlabel(r"$x_1$", fontsize=18)
ax.set_ylabel(r"$x_2$", fontsize=18)
ax.legend()

#### **符号计算** / **Symbolic Computation**

<!-- bilingual -->

In [ ]:
A = sympy.Matrix([[2, 3], [5, 4]])
b = sympy.Matrix([4, 3])
A, b

In [ ]:
A.rank()  # 满秩有解

In [ ]:
A.norm()  # 范数

In [ ]:
A.condition_number()

In [ ]:
sympy.N(A.condition_number(), 4)  # 等价于方法.evalf(4)

求解线性问题最直接的方法是计算矩阵$A$的逆矩阵，但这不是找到解向量$x$的最有效的方法。更好的方法是对矩阵$A$进行[$LU$分解](https://zh.m.wikipedia.org/wiki/LU%E5%88%86%E8%A7%A3)，即$A=LU$，其中$L$是下三角矩阵，$U$是上三角矩阵。

The most direct way to solve a linear problem is to compute the inverse of the matrix $A$, but this is not the most efficient method to find the solution vector $x$. A better approach is to perform an [$LU$ decomposition](https://en.wikipedia.org/wiki/LU_decomposition) of $A$, i.e. $A=LU$, where $L$ is a lower-triangular matrix and $U$ is an upper-triangular matrix.

在SymPy中，可以使用`sympy.Matrix`类的`LUdecomposition`方法进行符号$LU$分解。该方法会返回两个新的Matrix对象、$L$和$U$矩阵以及一个行交换矩阵。当我们想要求解$Ax=b$的方程时，不需要显示计算$L$和$U$矩阵，而是使用`LUsolve`方法。

In SymPy, you can use the `LUdecomposition` method of the `sympy.Matrix` class to perform a symbolic $LU$ decomposition. This method returns two new Matrix objects, the $L$ and $U$ matrices, as well as a row-permutation matrix. When we want to solve $Ax=b$, we do not need to compute the $L$ and $U$ matrices explicitly; instead we use the `LUsolve` method.

<!-- bilingual -->

In [ ]:
L, U, P = A.LUdecomposition()

In [ ]:
L, U

In [ ]:
L * U == A

In [ ]:
x = A.LUsolve(b)
x

#### **数值计算** / **Numerical Computation**

对于数值问题，可以使用SciPy线性代数模块的`la.lu`函数。该函数返回置换矩阵$P$以及$L$和$U$矩阵，是的$A=PLU$。与SymPy情况一样，我们不需要显示计算$L$和$U$矩阵，而是使用`la.solve`来求解线性方程组，该函数将矩阵$A$和向量$b$作为参数。

For numerical problems, use the `la.lu` function from SciPy's linear algebra module. This function returns the permutation matrix $P$ together with the $L$ and $U$ matrices such that $A=PLU$. As in the SymPy case, we do not need to compute $L$ and $U$ explicitly; we use `la.solve` to solve the linear system, passing in the matrix $A$ and the vector $b$.

<!-- bilingual -->

In [ ]:
A = np.array([[2, 3], [5, 4]])
b = np.array([4, 3])

In [ ]:
np.linalg.matrix_rank(A)

In [ ]:
np.linalg.norm(A)

In [ ]:
np.linalg.cond(A)

In [ ]:
P, L, U = la.lu(A)

In [ ]:
L, U

In [ ]:
P@L@U == A # 等价于 np.dot(P, np.dot(L, U))

In [ ]:
la.solve(A, b)

下面我们将演示符号方法和数值方法的区别，并且说明数值方法对大条件数的方程组很敏感。在这个例子中我们将求解的方程组如下：

Below we demonstrate the difference between symbolic and numerical methods, and show that numerical methods are sensitive to systems with large condition numbers. In this example, we will solve the following system:

$$ \begin{bmatrix}
1 & \sqrt{p}  \\
1 & \frac{1}{\sqrt{p}}
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2
\end{bmatrix}
=
\begin{bmatrix}
1 \\
2
\end{bmatrix}
$$

当$p$=1时，该方程组是奇异的。当$p$是1附近的值时，方程组是病态条件的。我们使用SymPy可以很容易找到解：

When $p=1$, the system is singular. When $p$ is near 1, the system is ill-conditioned. We can easily find the solution using SymPy:

<!-- bilingual -->

In [ ]:
p = sympy.symbols("p", positive=True)
A = sympy.Matrix([[1, sympy.sqrt(p)], [1, 1/sympy.sqrt(p)]])
b = sympy.Matrix([1, 2])
A.solve(b)

In [ ]:
sympy.simplify(_)  # 此处下划线`_`表示上次运行结果

符号解是精确的（如果能找到），数值解的误差是由浮点数引起的，本例中二者的差异与参数p的关系如下图所示:

The symbolic solution is exact (when one can be found); the error of the numerical solution comes from floating-point numbers. In this example, the relationship between their difference and the parameter p is shown in the figure below:

<!-- bilingual -->

In [ ]:
p = sympy.symbols("p", positive=True)
A = sympy.Matrix([[1, sympy.sqrt(p)], [1, 1/sympy.sqrt(p)]])
b = sympy.Matrix([1, 2])

# Solve symbolically
x_sym_sol = A.solve(b)
x_sym_sol.simplify()
x_sym_sol
Acond = A.condition_number().simplify()

# Function for solving numerically
AA = lambda p: np.array([[1, np.sqrt(p)], [1, 1/np.sqrt(p)]])
bb = np.array([1, 2])
x_num_sol = lambda p: np.linalg.solve(AA(p), bb)

# Graph the difference between the symbolic (exact) and numerical results.
p_vec = np.linspace(0.9, 1.1, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for n in range(2):
    x_sym = np.array([x_sym_sol[n].subs(p, pp).evalf() for pp in p_vec])
    x_num = np.array([x_num_sol(pp)[n] for pp in p_vec])
    axes[0].plot(p_vec, (x_num - x_sym)/x_sym, 'k')
axes[0].set_title("Error in solution\n(numerical - symbolic)/symbolic")
axes[0].set_xlabel(r'$p$', fontsize=18)

axes[1].plot(p_vec, [Acond.subs(p, pp).evalf() for pp in p_vec])
axes[1].set_title("Condition number")
axes[1].set_xlabel(r'$p$', fontsize=18)

### 矩形方程组 / Rectangular Systems

#### **欠定方程组** / **Underdetermined Systems**

欠定方程组的变量数比方程数多，解无法唯一确定，必须用自由变量来表示。通常需要用符号方法来处理。

An underdetermined system has more variables than equations; the solution cannot be uniquely determined and must be expressed with free variables. It usually needs to be handled by symbolic methods.

$$ \begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3
\end{bmatrix}
=
\begin{bmatrix}
7 \\
8
\end{bmatrix}
$$

<!-- bilingual -->

In [ ]:
x_vars = sympy.symbols("x, y, z")
A = sympy.Matrix([[1, 2, 3], [4, 5, 6]])
x = sympy.Matrix(x_vars)
b = sympy.Matrix([7, 8])
sympy.solve(A*x - b, x_vars)

#### **超定方程组** / **Overdetermined Systems**

对于超定方程组，方程的数量比未知变量更多。我们为方形方程组谈论中使用的示例方程组增加一个条件：

For overdetermined systems, the number of equations exceeds the number of unknowns. We add an additional condition to the example system used in our square-system discussion:

<!-- bilingual -->

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x1 = np.linspace(-4, 2, 100)

x2_1 = (4 - 2 * x1)/3
x2_2 = (3 - 5 * x1)/4
x2_3 = (7 + 3 * x1)/4

ax.plot(x1, x2_1, 'r', lw=2, label=r"$2x_1+3x_2-4=0$")
ax.plot(x1, x2_2, 'b', lw=2, label=r"$5x_1+4x_2-3=0$")
ax.plot(x1, x2_3, 'b', lw=2, label=r"$3x_1-7x_2+7=0$")

ax.set_xlabel(r"$x_1$", fontsize=18)
ax.set_ylabel(r"$x_2$", fontsize=18)
ax.legend()

三个直线之间不一定存在交点。我们拥有的约束条件比自由度多，这种情况下方程组通常没有精确解。此时，我们通常需要尝试为超定方程组寻找近似解。

There is not necessarily an intersection among three lines. We have more constraints than degrees of freedom, and in such cases the system usually has no exact solution. Here we typically need to try to find an approximate solution to the overdetermined system.

数据拟合就是这种情况的佐证：假设有一个模型，变量$y$是变量$x$的二次多项式，$y=A+bx+cx^2$。这里变量$y$是变量$x$的关系是非线性的，但是三个未知数$A$、$B$和$C$是线性的。因此，我们可以将该模型写成一个线性方程组。如果我们采集了m组数据$\{(x_i, y_i)\}^m_{i=1}$，就可以将该模型写成如下$m×3$的方程组：

Data fitting is an example of this scenario: suppose there is a model where the variable $y$ is a quadratic polynomial of the variable $x$, $y=A+Bx+Cx^2$. Here the relationship between $y$ and $x$ is nonlinear, but the three unknowns $A$, $B$, and $C$ are linear. Therefore, we can write this model as a linear system. If we have collected $m$ data points $\{(x_i, y_i)\}^m_{i=1}$, we can write the model as the following $m \times 3$ system:

<!-- bilingual -->

$$ \begin{bmatrix}
1 & x_1 & x_1^2 \\
\vdots & \vdots & \vdots \\
1 & x_m & x_m^2
\end{bmatrix}
\begin{bmatrix}
A \\
B \\
C
\end{bmatrix}
=
\begin{bmatrix}
y_1 \\
\vdots \\
y_m
\end{bmatrix}
$$

<!-- bilingual -->

如果$m>3$，那么通常没有精确解，需要引入近似解，为超定方程组$Ax \approx b$给出最佳拟合。针对这种方程组最佳拟合的一种自然定义是最小误差平方和$\min_x  \sum_{i=1}^{m} r_i^2 $，其中$r= b-Ax$是残差向量。在SymPy中，可以使用`solve_least_squares`方法来求解超定方程组的最小二乘解。对于数值问题，可以使用SciPy中的`la.lstsq`函数。

If $m>3$, there is generally no exact solution, and an approximate solution must be introduced to give a best fit for the overdetermined system $Ax \approx b$. A natural definition of best fit for such a system is the minimum sum of squared errors, $\min_x \sum_{i=1}^{m} r_i^2$, where $r = b - Ax$ is the residual vector. In SymPy, you can use the `solve_least_squares` method to obtain the least-squares solution of an overdetermined system. For numerical problems, you can use SciPy's `la.lstsq` function.

<!-- bilingual -->

下面的代码演示了使用SciPy的`la.lstsq`函数进行数据拟合。我们首先定义模型的真实参数，然后在真实的模型关系中加入随机噪声模拟测量数据。最后，我们使用`la.lstsq`函数对参数A、B和C进行拟合。

The code below demonstrates data fitting using SciPy's `la.lstsq` function. We first define the true parameters of the model and then add random noise to the true model relationship to simulate measurement data. Finally, we use `la.lstsq` to fit the parameters A, B, and C.

<!-- bilingual -->

In [ ]:
np.random.seed(1234)

# define true model parameters
x = np.linspace(-1, 1, 100)
a, b, c = 1, 2, 3
y_exact = a + b * x + c * x**2

# simulate noisy data points
m = 100
X = 1 - 2 * np.random.rand(m)
Y = a + b * X + c * X**2 + np.random.randn(m)

# fit the data to the model using linear least square
A = np.vstack([X**0, X**1, X**2])  # see np.vander for alternative
sol, r, rank, sv = la.lstsq(A.T, Y)
print(sol)
y_fit = sol[0] + sol[1] * x + sol[2] * x**2
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(X, Y, 'go', alpha=0.5, label='Simulated data')
ax.plot(x, y_exact, 'k', lw=2, label='True value $y = 1 + 2x + 3x^2$')
ax.plot(x, y_fit, 'b', lw=2, label='Least square fit')
ax.set_xlabel(r"$x$", fontsize=18)
ax.set_ylabel(r"$y$", fontsize=18)
ax.legend(loc=2)

想让数据与模型很好地拟合，显然要求用于描述数据的模型能够很好地与生成数据地过程对应，并能够有效地去除可能存在随机误差。我们把前一个示例中的数据你和到一个线性模型和一个高阶多项式。前者是欠拟合，因为使用的模型过于简单。后者是过拟合，不仅拟合了数据的潜在趋势，也拟合了测量噪声。

To fit data well to a model, the model used to describe the data must obviously correspond well to the process that generated the data and be able to effectively reject any random errors that may be present. We fit the data from the previous example to a linear model and a high-order polynomial. The former is underfitting, because the model is too simple. The latter is overfitting, capturing not only the underlying trend of the data but also the measurement noise.

<!-- bilingual -->

In [ ]:
# fit the data to the model using linear least square: 
# 1st order polynomial
A = np.vstack([X**n for n in range(2)])
sol, r, rank, sv = la.lstsq(A.T, Y)
y_fit1 = sum([s * x**n for n, s in enumerate(sol)])

# 15th order polynomial
A = np.vstack([X**n for n in range(16)])
sol, r, rank, sv = la.lstsq(A.T, Y)
y_fit15 = sum([s * x**n for n, s in enumerate(sol)])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(X, Y, 'go', alpha=0.5, label='Simulated data')
ax.plot(x, y_exact, 'k', lw=2, label='True value $y = 1 + 2x + 3x^2$')
ax.plot(x, y_fit1, 'b', lw=2, label='Least square fit [1st order]')
ax.plot(x, y_fit15, 'm', lw=2, label='Least square fit [15th order]')
ax.set_xlabel(r"$x$", fontsize=18)
ax.set_ylabel(r"$y$", fontsize=18)
ax.legend(loc=2)

## 特征值问题 / Eigenvalue Problems

---

一种非常重要的特殊方程是[特征值方程](https://zh.wikipedia.org/wiki/特征值和特征向量)$Ax=\lambda x$，其中$A$是一个N×N的方阵，$x$是未知向量，$\lambda$是未知标量。这里$x$是矩阵$A$的特征向量，$\lambda$是矩阵$A$的特征值。特征向量$x$经过矩阵$A$的线性变换后，得到的新向量仍然与原来的$x$保持在同一条直线上，但其长度或方向也许会改变。量子力学中的薛定谔方程就是特征值方程。

A very important special kind of equation is the [eigenvalue equation](https://en.wikipedia.org/wiki/Eigenvalues_and_eigenvectors) $Ax=\lambda x$, where $A$ is an $N \times N$ square matrix, $x$ is an unknown vector, and $\lambda$ is an unknown scalar. Here $x$ is an eigenvector of $A$, and $\lambda$ is an eigenvalue of $A$. After the eigenvector $x$ is transformed linearly by the matrix $A$, the new vector remains on the same line as the original $x$, but its length or direction may change. The Schrödinger equation in quantum mechanics is an eigenvalue equation.

<!-- bilingual -->

求解这类特征问题的标准方式是将方程写为$(A-I\lambda)x=0$的形式。如果存在非平凡解$x\neq0$，那么矩阵$A-I\lambda$必须是奇异的，其行列式必须为0，即$\det(A-I\lambda)x=0$。这将得到一个N阶多项式（特征多项式），它的N个跟会给出N个特征值。一旦特征值已知，就可以使用标准的前向替换法求解该特征值对应的特征向量。

The standard way to solve such eigenvalue problems is to write the equation in the form $(A - I\lambda)x = 0$. For a non-trivial solution $x \neq 0$ to exist, the matrix $A - I\lambda$ must be singular, meaning its determinant must be zero, i.e. $\det(A - I\lambda) = 0$. This yields an N-th order polynomial (the characteristic polynomial) whose N roots give the N eigenvalues. Once the eigenvalues are known, the eigenvector corresponding to an eigenvalue can be found by standard forward substitution.

<!-- bilingual -->

#### **符号计算** / **Symbolic Computation**

在SymPy中，可以使用Matrix类的`eigenvals`和`eigenvects`方法，求解具有符号元素的矩阵的特征值和特征向量。

In SymPy, the `eigenvals` and `eigenvects` methods of the Matrix class can be used to find eigenvalues and eigenvectors of matrices with symbolic entries.

<!-- bilingual -->

In [ ]:
eps, delta = sympy.symbols("epsilon, delta")
H = sympy.Matrix([[eps, delta], [delta, -eps]])
H

In [ ]:
H.eigenvals()

In [ ]:
H.eigenvects()

`eigenvals`方法的返回值是一个字典，每个特征值是一个键，对应的值是该特征值的重数（multiplicity）。这里的两个特征值重数都是1。`eigenvects`方法返回的是一个列表，其中每一个元素是一个元组，这个元组包含特征值、特征值的重数和特征向量列表。

The return value of the `eigenvals` method is a dictionary where each eigenvalue is a key and the corresponding value is its multiplicity. Here both eigenvalues have multiplicity 1. The `eigenvects` method returns a list in which each element is a tuple containing an eigenvalue, its multiplicity, and a list of eigenvectors.

<!-- bilingual -->

In [ ]:
(_, _, evec1), (_, _, evec2) = H.eigenvects()

In [ ]:
(evec1[0].T * evec2[0]).expand()

#### **数值计算** / **Numerical Computation**

对于较大的方程组，必须使用完全数值化的方法，可以使用SciPy线性代数包中的`la.eigvals`和`la.eig`函数。Hermitian矩阵和实数对称矩阵具有实数特征值，对于这类矩阵，使用`la.eigvalsh`和`la.eigh`函数更有优势。

For larger systems, a fully numerical method must be used. SciPy's linear algebra package provides `la.eigvals` and `la.eig`. For Hermitian or real symmetric matrices, which have real eigenvalues, `la.eigvalsh` and `la.eigh` are preferable.

<!-- bilingual -->

In [ ]:
A = np.array([[1, 3, 5], [3, 5, 3], [5, 3, 9]])
evals, evecs = la.eig(A)

In [ ]:
evals

In [ ]:
evecs

In [ ]:
la.eigh(A)

## 非线性方程 / Nonlinear Equations

---

根据定义，线性方程满足可加性$f(x+y)=f(x)+f(y)$和齐次性$f(\alpha x)=\alpha f(x)$。非线性函数不满足这些条件。在自然科学和工程科学中，很多系统本质上是非线性的。

By definition, linear equations satisfy additivity $f(x+y) = f(x) + f(y)$ and homogeneity $f(\alpha x) = \alpha f(x)$. Nonlinear functions do not satisfy these properties. In the natural and engineering sciences, many systems are inherently nonlinear.

<!-- bilingual -->

### 单变量非线性方程 / Univariate Nonlinear Equations

与线性方程不同，没有通用的方法得到非线性方程的一个或多个解。从分析上看，只有特定形式的方程额可以精确求解。例如，对于四阶及以下多项式，可以通过解析方法进行求解。另外，包含三角函数以及其它基本函数的一些方程也可以通过解析的方法进行求解。在Sympy中，可以使用sympy.solve函数对很多可以解析的单变量非线性方程进行求解。

Unlike linear equations, there is no universal method to obtain one or more solutions of a nonlinear equation. Analytically, only equations of specific forms can be solved exactly. For example, polynomials of degree four or less can be solved analytically. In addition, some equations involving trigonometric functions or other elementary functions can also be solved analytically. In SymPy, the sympy.solve function can solve many univariate nonlinear equations that admit analytical solutions.

<!-- bilingual -->

In [ ]:
x, a, b, c = sympy.symbols("x, a, b, c")
sympy.solve(a + b*x + c*x**2, x)

一般情况下，非线性方程是无法解析求解的。例如$\sin x=x$是超越方程，不存在代数解。

In general, nonlinear equations cannot be solved analytically. For example, $\sin x = x$ is a transcendental equation and has no algebraic solution.

<!-- bilingual -->

In [ ]:
try:
    sympy.solve(sympy.sin(x) - x, x)
except NotImplementedError:
    print("No algorithms are implemented to solve equation")

在这种情况下，需要使用各种数值方法。首先绘制图形，这可以帮助我们得到有关方程解的数量以及大概位置的线索。在使用数值方法寻找方程近似跟的时候，通常需要这些信息。

In such cases, we must resort to various numerical methods. First, plotting the figure can give us hints about the number of solutions of the equation and their approximate locations. This information is usually needed when using numerical methods to find approximate roots.

<!-- bilingual -->

In [ ]:
x = np.linspace(-2, 2, 1000)

# four examples of nonlinear functions
f1 = x**2 - x - 1
f2 = x**3 - 3 * np.sin(x)
f3 = np.exp(x) - 2
f4 = 1 - x**2 + np.sin(50 / (1 + x**2))

# plot each function
fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharey=True)

for n, f in enumerate([f1, f2, f3, f4]):
    axes[n].plot(x, f, lw=1.5)
    axes[n].axhline(0, ls=':', color='k')
    axes[n].set_ylim(-5, 5)
    axes[n].set_xticks([-2, -1, 0, 1, 2])
    axes[n].set_xlabel(r'$x$', fontsize=18)

axes[0].set_ylabel(r'$f(x)$', fontsize=18)

titles = [r'$f(x)=x^2-x-1$', r'$f(x)=x^3-3\sin(x)$',
          r'$f(x)=\exp(x)-2$', r'$f(x)=\sin\left(50/(1+x^2)\right)+1-x^2$']
for n, title in enumerate(titles):
    axes[n].set_title(title)

为了找到方程解的近似位置，可以从众多[数值求解方法](https://zh.wikipedia.org/zh-hans/数值分析)中选择一种。这些方法通常使用**迭代**的方式，计算函数在某个连续区域的值，直到算法收敛到某个所需的精度。有两种标准的方法演示了数字求根方法的基本思想：[二分法](https://zh.wikipedia.org/zh-hans/二分法_(數學))和[牛顿法](https://zh.wikipedia.org/zh-hans/牛顿法)。

To find approximate locations of solutions, one of many [numerical methods](https://en.wikipedia.org/wiki/Numerical_analysis) can be chosen. These methods typically use an **iterative** approach, evaluating the function over some continuous region until the algorithm converges to the desired precision. Two standard methods illustrate the basic idea of numerical root-finding: [bisection method](https://en.wikipedia.org/wiki/Bisection_method) and [Newton's method](https://en.wikipedia.org/wiki/Newton%27s_method).

<!-- bilingual -->

#### **二分法** / **Bisection Method**

<!-- bilingual -->

In [ ]:
# define a function, desired tolerance and starting interval [a, b]
f = lambda x: np.exp(x) - 2
tol = 0.1  # 期望精度
a, b = -2, 2  # 初始尝试解
x = np.linspace(-2.1, 2.1, 1000)

# graph the function f
fig, ax = plt.subplots(1, 1, figsize=(12, 4))

ax.plot(x, f(x), lw=1.5)
ax.axhline(0, ls=':', color='k')
ax.set_xticks([-2, -1, 0, 1, 2])
ax.set_xlabel(r'$x$', fontsize=18)
ax.set_ylabel(r'$f(x)$', fontsize=18)

# find the root using the bisection method and visualize
# the steps in the method in the graph
fa, fb = f(a), f(b)

ax.plot(a, fa, 'ko')
ax.plot(b, fb, 'ko')
ax.text(a, fa + 0.5, r"$a$", ha='center', fontsize=18)
ax.text(b, fb + 0.5, r"$b$", ha='center', fontsize=18)

n = 1
while b - a > tol:  # 迭代求解
    m = a + (b - a)/2
    fm = f(m)
    ax.plot(m, fm, 'ko')
    ax.text(m, fm - 0.5, r"$m_%d$" % n, ha='center')
    n += 1
    if np.sign(fa) == np.sign(fm):  # 判断中值符号
        a, fa = m, fm
    else:
        b, fb = m, fm

ax.plot(m, fm, 'r*', markersize=10)
ax.annotate("Root approximately at %.3f" % m,
            fontsize=14, family="serif",
            xy=(a, fm), xycoords='data',
            xytext=(-150, +50), textcoords='offset points', 
            arrowprops=dict(arrowstyle="->", connectionstyle="arc3, rad=-.5"))

ax.set_title("Bisection method")


#### 牛顿法 / Newton's Method

牛顿法的收敛速度比二分法更快。二分法只使用每个点函数值的符号，牛顿法使用$f(x)$的一阶泰勒展开式$f(x)=f(x_0)+(x-x_0)f'(x_0)$来近似函数$f(x)$。一阶泰勒展开式是一个线性函数，很容易找到它的根$x_0-\frac{f(x_0)}{f'(x_0)}$。需要注意的潜在问题是，如果存在某些点$x_k$使得$f'(x_k)=0$，该方法会失效。下面的例子演示了如何使用牛顿法求方程'$\exp(x)-2=0$的根，其中使用了SymPy来计算$f(x)$的导数。

Newton's method converges faster than the bisection method. The bisection method only uses the sign of the function value at each point; Newton's method uses the first-order Taylor expansion of $f(x)$, $f(x) = f(x_0) + (x - x_0) f'(x_0)$, to approximate $f(x)$. The first-order Taylor expansion is a linear function, and its root is easily found to be $x_0 - \frac{f(x_0)}{f'(x_0)}$. One potential problem is that if there is some point $x_k$ where $f'(x_k) = 0$, the method fails. The example below shows how to use Newton's method to find the root of the equation $\exp(x) - 2 = 0$, using SymPy to compute the derivative of $f(x)$.

<!-- bilingual -->

In [ ]:
# define a function, desired tolerance and starting point xk
tol = 0.01
xk = 2

s_x = sympy.symbols("x")
s_f = sympy.exp(s_x) - 2

f = lambda x: sympy.lambdify(s_x, s_f, 'numpy')(x)
fp = lambda x: sympy.lambdify(s_x, sympy.diff(s_f, s_x), 'numpy')(x)

x = np.linspace(-1, 2.1, 1000)

# setup a graph for visualizing the root finding steps
fig, ax = plt.subplots(1, 1, figsize=(12,4))

ax.plot(x, f(x))
ax.axhline(0, ls=':', color='k')

# repeat Newton's method until convergence to the desired tolerance has been reached
n = 0
while f(xk) > tol:
    xk_new = xk - f(xk) / fp(xk)

    ax.plot([xk, xk], [0, f(xk)], color='k', ls=':')
    ax.plot(xk, f(xk), 'ko')
    ax.text(xk, -.5, r'$x_%d$' % n, ha='center')
    ax.plot([xk, xk_new], [f(xk), 0], 'k-')

    xk = xk_new
    n += 1

ax.plot(xk, f(xk), 'r*', markersize=15)
ax.annotate("Root approximately at %.3f" % xk,
            fontsize=14, family="serif",
            xy=(xk, f(xk)), xycoords='data',
            xytext=(-150, +50), textcoords='offset points', 
            arrowprops=dict(arrowstyle="->", connectionstyle="arc3, rad=-.5"))

ax.set_title("Newton's method")
ax.set_xticks([-1, 0, 1, 2])

牛顿法存在的潜在问题是，需要在每次迭代中计算函数值和函数的导数值。在上个示例中，我们使用SymPy通过符号来计算导数。在全数值的实现中，这显然是不可能的。此外，工程和科学计算中$f(x)$的表达式未必是已知的。所以我们需要用数值方法求导数的近似值。牛顿法的一种变体割线法（secant method），使用函数的前两个计算值来获得函数当前值的线性近似值，该线性近似值可用于计算根的最新估计值。割线法的迭代公式是：

A potential problem with Newton's method is that both the function value and the derivative must be computed at every iteration. In the previous example, we used SymPy to compute the derivative symbolically. In a fully numerical implementation, that is clearly not possible. Moreover, the analytic expression of $f(x)$ may not be known in engineering and scientific computation. So we need to approximate the derivative numerically. A variant of Newton's method, the secant method, uses the two previously computed function values to obtain a linear approximation of the current function value, and this linear approximation is used to update the estimate of the root. The iteration formula of the secant method is:

$$x_{k+1} = x_k - f(x_k) \frac{x_k - x_{k-1}}{f(x_k)-f(x_{k-1})}$$

更先进的数值求根方法通常是二分法或者牛顿法的变形或者结合，例如函数的高阶插值。

More advanced numerical root-finding methods are typically variants or combinations of bisection or Newton's method, such as those that use higher-order interpolation of the function.

<!-- bilingual -->

SciPy的optimize模块提供了多个用于数值求根的函数。`optimize.bisect`和`optimize.newton`函数实现了变体形式的二分法和牛顿法。`optimize.bisect`有三个参数：第一个函数是需要求解的方程的数学函数；第二个和第三个参数是二分法的初始区间的下限和上限。需要注意，函数在初始区间上下限处的值的符号必须不同。

SciPy's optimize module provides several functions for numerical root-finding. `optimize.bisect` and `optimize.newton` implement variants of the bisection method and Newton's method. `optimize.bisect` takes three arguments: the first is the mathematical function whose root is sought; the second and third are the lower and upper bounds of the initial bracketing interval. Note that the function values at the bounds of the initial interval must have opposite signs.

<!-- bilingual -->

In [ ]:
f = lambda x: np.exp(x) - 2
optimize.bisect(f, -2, 2)

`optimize.newton`函数的第一个参数也是待求解的函数，第二个参数是函数解的初始猜测值。还有一个可选的参数fprime，用于指定函数的导数。如果给出了fprime，则使用牛顿法，否则使用割线法。

The first argument of `optimize.newton` is also the function whose root is sought, and the second is an initial guess for the root. There is also an optional argument `fprime` for specifying the derivative of the function. If `fprime` is provided, Newton's method is used; otherwise the secant method is used.

<!-- bilingual -->

In [ ]:
x_guess = 2
f = lambda x: np.exp(x) - 2
fprime = lambda x: np.exp(x)
optimize.newton(f, x_guess, fprime)  # 牛顿法

In [ ]:
optimize.newton(f, x_guess)  # 割线法 

SciPy的optimize模块还提供了其它函数。特别是`optimize.brentq`和`optimize.brenth`函数，它们都是二分法的高级变体。[Brent方法](https://en.wikipedia.org/wiki/Brent%27s_method)通常被认为是首选的求根方法。

SciPy's optimize module also provides other functions. In particular, `optimize.brentq` and `optimize.brenth` are advanced variants of the bisection method. [Brent's method](https://en.wikipedia.org/wiki/Brent%27s_method) is generally considered the preferred root-finding method.

<!-- bilingual -->

In [ ]:
optimize.brentq(f, -2, 2)

### 非线性方程组 / Systems of Nonlinear Equations

非线性方程组无法写成矩阵乘法的形式。我们可以将多元非线性方程组表示成向量值函数（vector-valued function），如$f:\mathbb{R}^N \rightarrow \mathbb{R}^N $，这表示一个N维向量映射到另一个N维向量。多变量方程组相比单变量方程更难以求解，没有严格保证收敛到某个解的方法。在后续的章节中，我们将尝试用神经网络求解非线性方程组。

Systems of nonlinear equations cannot be written as matrix multiplications. We can represent a multivariate nonlinear system as a vector-valued function $f: \mathbb{R}^N \rightarrow \mathbb{R}^N$, which maps an N-dimensional vector to another N-dimensional vector. Multivariate systems are harder to solve than univariate equations, and there is no method that strictly guarantees convergence to a solution. In later chapters, we will attempt to solve systems of nonlinear equations using neural networks.

<!-- bilingual -->

当变量数目只有2个时，我们仍然可以使用作图的方法，大致确定解的范围。例如，如下包含$x$和$y$两个变量的非线性方程组：
$$\begin{cases} x + y^2 = 4 \\
e^x + xy = 3 \end{cases}$$

When there are only two variables, we can still use a graphical approach to roughly determine the location of solutions. For example, the following nonlinear system with variables $x$ and $y$:
$$\begin{cases} x + y^2 = 4 \\
e^x + xy = 3 \end{cases}$$

<!-- bilingual -->

In [ ]:
# 交互式绘图
%matplotlib widget

x = np.linspace(0, 1, 100)
y = np.linspace(1, 2, 100)
X, Y = np.meshgrid(x, y)
Z1 = X + Y**2 - 4
Z2 = np.exp(X) + X*Y - 3
Z3 = np.zeros_like(Z1)

# graph the function f
fig, ax = plt.subplots(1, 1, figsize=(6, 6), subplot_kw={'projection': '3d'})

ax.plot_wireframe(X, Y, Z1, rstride=4, cstride=4, color="red", alpha=0.3)
ax.plot_wireframe(X, Y, Z2, rstride=4, cstride=4, color="green", alpha=0.3)
ax.plot_wireframe(X, Y, Z3, rstride=4, cstride=4, color="darkgrey")

ax.set_xlabel(r'$X$', fontsize=18)
ax.set_ylabel(r'$Y$', fontsize=18)
ax.set_zlabel(r"$Z$", fontsize=16)

在这个示例中，通过变量替换$x=4-y^2$，多变量方程组可以化简为一个单变量方程$e^{4-y^2} + (4-y^2)y - 3= 0$。

In this example, using the substitution $x = 4 - y^2$, the multivariate system can be reduced to a single equation in one variable: $e^{4 - y^2} + (4 - y^2)y - 3 = 0$.

<!-- bilingual -->

In [ ]:
f = lambda y: np.exp(4-y**2) + (4-y**2)*y - 3
sol_y = optimize.brentq(f, 0, 2)
sol_x = 4 - sol_y**2
sol_x , sol_y

当变量的数目增加时，计算的难度也会增加。求解单变量方程的方法并不能直接推广到多变量的情况。二分法不能直接推广到多变量方程组，牛顿法可以用于多变量问题。在这种情况下，迭代方程是$x_(k+1)=x_k-J_f(x_k)^{-1}f(x_k)$，其中$J_f(x_k)$是函数$f(x)$的[雅可比矩阵](https://zh.m.wikipedia.org/zh-hans/雅可比矩阵)（Jacobian Matrix），其中的元素是$[J_f(x_k)]_{ij}=\delta f_i(x_k) / \delta x_j$。雅可比矩阵类似于多元函数的导数。该方法只需要求解线性方程组$J_f(x_k)\delta x_k = -f(x_k)$，然后使用$x_{k+1}=x_k+\delta x_k$，而不需要求雅可比矩阵的逆。

As the number of variables grows, the difficulty of the computation also grows. Methods for solving univariate equations do not directly generalize to multivariate cases. The bisection method does not directly generalize to multivariate systems, while Newton's method can be applied to multivariate problems. In this case, the iteration formula is $x_{k+1} = x_k - J_f(x_k)^{-1} f(x_k)$, where $J_f(x_k)$ is the [Jacobian matrix](https://en.wikipedia.org/wiki/Jacobian_matrix_and_determinant) of $f(x)$, whose entries are $[J_f(x_k)]_{ij} = \partial f_i(x_k) / \partial x_j$. The Jacobian matrix is analogous to the derivative of a multivariate function. The method only needs to solve the linear system $J_f(x_k) \delta x_k = -f(x_k)$ and then use $x_{k+1} = x_k + \delta x_k$, avoiding the need to invert the Jacobian.

与单变量方程组中牛顿法的割线法变体一样，也有多变量函数的变体，可以根据函数先前的计算来估计函数的当前值，从而避免计算雅可比矩阵。[Broyden法](https://en.wikipedia.org/wiki/Broyden%27s_method)就是这类多变量方程组的割线更新法的典型例子。

As with the secant variant of Newton's method in the univariate case, there are multivariate variants that estimate the current function value using previous function evaluations, avoiding the computation of the Jacobian matrix. [Broyden's method](https://en.wikipedia.org/wiki/Broyden%27s_method) is a typical secant-update method for multivariate systems.

在SciPy模块中，`broyden1`和`broyden2`是两个使用不同Jacobian近似值实现Broyden法的函数；而`optimize.fsolve`则提供了一种类似牛顿法的实现，该函数由一个可选参数用于指定雅各可比矩阵。

In the SciPy module, `broyden1` and `broyden2` are two functions that implement Broyden's method using different Jacobian approximations; `optimize.fsolve` provides a Newton-like implementation that accepts an optional argument for specifying the Jacobian matrix.

<!-- bilingual -->

In [ ]:
def f(x):
    return [x[0] + x[1]**2 - 4, np.exp(x[0]) + x[0]*x[1] - 3]

sol1 = optimize.broyden1(f, [1, 1])
print(sol1)
sol2 = optimize.broyden2(f, [1, 1])
print(sol2)
sol3 = optimize.fsolve(f, [1, 1])
print(sol3)

与单变量非线性牛顿方程的牛顿法一样，解的初始猜测值非常重要，不同的初始猜测值可能会导致找到不同的方程解。求解非线性方程是一项很复杂的工作，各种类型的可视化通常对于构建特定问题特征的理解非常有用。

As with Newton's method for univariate nonlinear equations, the initial guess of the solution is very important; different initial guesses may lead to different solutions. Solving nonlinear equations is a complex task, and various kinds of visualization are often very helpful for building intuition about a specific problem.

<!-- bilingual -->

#### 超定非线性方程组 / Overdetermined Nonlinear Systems

上述示例中，变量和约束条件一致。如果非线形问题的约束条件多于变量数目，与线性方程组类似，精确解可能不存在，此时需要寻找方程组的最优近似解。[高斯牛顿迭代法](https://en.wikipedia.org/wiki/Gauss%E2%80%93Newton_algorithm)是一个常用选择。

In the above example, the numbers of variables and constraints matched. If a nonlinear problem has more constraints than variables, then, as with linear systems, an exact solution may not exist, and we need to find a best approximate solution. The [Gauss-Newton algorithm](https://en.wikipedia.org/wiki/Gauss%E2%80%93Newton_algorithm) is a common choice.

<!-- bilingual -->